# Digital Twin Extraction powered by CASCADES

This notebook runs the CASCADES Server (Qwen3-4B with NF4 quantization and CASCADES adapters) and processes your personal Google Takeout data chunks into Neo4j Cypher statements, while simultaneously feeding the structured facts into Hemisphere B for parametric memory consolidation.

It is designed to run on a **T4 GPU** (free tier Google Colab).

## 1. Setup Environment

In [ ]:
!pip install -q torch torchvision torchaudio
!pip install -q transformers accelerate bitsandbytes tokenizers
!pip install -q fastapi uvicorn sse-starlette requests

## 2. Mount Drive & Extract Model

**Prerequisites** (in your Google Drive root):
1. `abliteratedqwen3.zip` — the Qwen3-4B abliterated model
2. `takeout_chunks_32k/` — your Google Takeout data chunks
3. `CASCADES_Colab_Minimal.zip` — the CASCADES source code

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile, shutil

MODEL_PATH = "/content/abliterated"
CHUNKS_DIR = "/content/drive/MyDrive/takeout_chunks_32k"
CYPHER_DIR = "/content/drive/MyDrive/cypher_output"
os.makedirs(CYPHER_DIR, exist_ok=True)

# ── Extract model to fast local NVMe ──
if not os.path.exists(os.path.join(MODEL_PATH, 'config.json')):
    print("Extracting model to local Colab storage...")
    os.makedirs(MODEL_PATH, exist_ok=True)
    zip_path = "/content/drive/MyDrive/abliteratedqwen3.zip"
    if not os.path.exists(zip_path):
        raise FileNotFoundError(f"{zip_path} not found in Google Drive.")
    
    # Copy locally first for speed
    local_zip = "/content/temp_model.zip"
    if not os.path.exists(local_zip):
        print("  Copying zip locally (using cp to avoid FUSE errors)...")
        import subprocess
        subprocess.run(["cp", zip_path, local_zip], check=True)
    
    # Handle nested root folder in zip
    with zipfile.ZipFile(local_zip, "r") as zf:
        members = zf.namelist()
        top_dirs = {m.split("/")[0] for m in members if "/" in m}
        if len(top_dirs) == 1:
            top = top_dirs.pop()
            print(f"  Stripping root folder '{top}/' from zip...")
            for member in members:
                rel = os.path.relpath(member, top)
                if rel == ".": continue
                dest = os.path.join(MODEL_PATH, rel)
                if member.endswith("/"):
                    os.makedirs(dest, exist_ok=True)
                else:
                    os.makedirs(os.path.dirname(dest), exist_ok=True)
                    with zf.open(member) as src, open(dest, "wb") as dst:
                        dst.write(src.read())
        else:
            zf.extractall(MODEL_PATH)
    os.remove(local_zip)
    print("Model extracted!")
else:
    print(f"Model already extracted at {MODEL_PATH}")

# Verify
model_files = os.listdir(MODEL_PATH)
print(f"Model dir contents ({len(model_files)} files): {model_files[:10]}")
assert 'config.json' in model_files, "ERROR: config.json not found in model dir!"

## 3. Install CASCADES Source Code

In [ ]:
import os, sys

REPO_DIR = "/content/CASCADES_REPO"
os.makedirs(REPO_DIR, exist_ok=True)

# Copy and extract
!cp /content/drive/MyDrive/CASCADES_Colab_Minimal.zip /content/CASCADES_REPO/
%cd /content/CASCADES_REPO/
!unzip -o -q CASCADES_Colab_Minimal.zip

sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

# Verify critical files exist
for f in ['app/server.py', 'app/model_loader.py', 'cascades/adapters.py', 'cascades/injection.py']:
    assert os.path.exists(f), f"MISSING: {f}"
    print(f"  ✓ {f}")
print("CASCADES source code ready.")

## 4. GPU & Import Diagnostics

This cell validates the GPU, CUDA, and all critical imports **before** attempting to start the server. If something fails here, we fix it before wasting time on a 2-minute model load.

In [ ]:
print("=" * 60)
print("GPU & IMPORT DIAGNOSTICS")
print("=" * 60)

# 1. GPU check
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
    print(f"VRAM used: {torch.cuda.memory_allocated() / 1024**2:.0f} MB")
else:
    print("WARNING: No GPU detected! Go to Runtime > Change runtime type > T4 GPU")

# 2. Critical imports
errors = []
for mod_name in ['transformers', 'accelerate', 'bitsandbytes', 'fastapi', 'uvicorn', 'sse_starlette']:
    try:
        __import__(mod_name)
        print(f"  ✓ {mod_name}")
    except ImportError as e:
        errors.append(f"  ✗ {mod_name}: {e}")
        print(f"  ✗ {mod_name}: {e}")

# 3. CASCADES module imports
print("\nCASCADES modules:")
for mod_name in ['cascades.config', 'cascades.adapters', 'cascades.injection', 'cascades.math_ops',
                  'app.conversation_store', 'app.self_synthesizer', 'app.model_loader', 'app.dream_cycle']:
    try:
        __import__(mod_name)
        print(f"  ✓ {mod_name}")
    except Exception as e:
        errors.append(f"  ✗ {mod_name}: {e}")
        print(f"  ✗ {mod_name}: {type(e).__name__}: {e}")

if errors:
    print(f"\n⚠️ {len(errors)} IMPORT ERRORS — fix these before proceeding!")
else:
    print("\n✅ All imports OK!")

## 5. Start CASCADES Server (In-Process)

This runs the server **in-process** using a background thread instead of a subprocess. This means:
- Errors show up immediately in the cell output (no hidden stderr)
- OOM crashes are visible
- You can inspect `model` and `dream` objects directly

In [ ]:
import threading
import time
import logging
import requests
import torch

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(name)s %(levelname)s: %(message)s')

CASCADES_URL = "http://127.0.0.1:8000"
SERVER_HOST = "127.0.0.1"
SERVER_PORT = 8000
ADAPTER_RANK = 32

# ── Step 1: Load the model directly ──
print("="*60)
print(f"LOADING MODEL: {MODEL_PATH}")
print(f"GPU VRAM before: {torch.cuda.memory_allocated()/1024**2:.0f} MB")
print("="*60)

from app.model_loader import CASCADESModel

cascades_model = CASCADESModel(
    model_id=MODEL_PATH,
    adapter_weights=None,
    rank=ADAPTER_RANK,
)

try:
    cascades_model.load()
    print(f"\n✅ Model loaded! VRAM: {torch.cuda.memory_allocated()/1024**2:.0f} MB")
except Exception as e:
    print(f"\n🚨 MODEL LOAD FAILED: {type(e).__name__}: {e}")
    import traceback
    traceback.print_exc()
    raise  # Stop here so user can see the error

In [ ]:
# ── Step 2: Initialize Dream Cycle ──
from app.dream_cycle import DreamCycle

dream_cycle = DreamCycle(
    model=cascades_model.model,
    tokenizer=cascades_model.tokenizer,
    brain_lock=cascades_model._lock,
    min_examples=3,
)
print("🧠 Hemisphere B (Dream Cycle) initialized")

# ── Step 3: Wire up the server globals ──
import app.server as srv
srv.model = cascades_model
srv.dream = dream_cycle

print("Server globals wired up.")

In [ ]:
# ── Step 3: Start uvicorn in a background thread ──
import uvicorn

def _run_server():
    """Run uvicorn in background. Errors print to this cell's output."""
    try:
        uvicorn.run(srv.app, host=SERVER_HOST, port=SERVER_PORT, log_level="info")
    except Exception as e:
        print(f"\n🚨 SERVER CRASHED: {type(e).__name__}: {e}")
        import traceback
        traceback.print_exc()

server_thread = threading.Thread(target=_run_server, daemon=True)
server_thread.start()

# Wait for it to come up
print("Waiting for server to bind...", end="", flush=True)
for i in range(30):
    time.sleep(1)
    try:
        resp = requests.get(f"{CASCADES_URL}/v1/memory/stats", timeout=2)
        if resp.status_code == 200:
            print(f"\n\n✅ CASCADES Server is UP at {CASCADES_URL}")
            print(f"Stats: {resp.json()}")
            break
    except requests.ConnectionError:
        print(".", end="", flush=True)
else:
    print("\n⚠️ Server didn't respond after 30s. Check output above for errors.")
    if not server_thread.is_alive():
        print("🚨 Server thread has died!")

## 6. Run The Extraction Pipeline

In [ ]:
import re, os

script_path = os.path.join(REPO_DIR, 'local_extract_cascades.py')
if os.path.exists(script_path):
    with open(script_path, 'r', encoding='utf-8') as f:
        content = f.read()
    
    content = re.sub(
        r'CHUNKS_DIR = r".*?"',
        f'CHUNKS_DIR = r"{CHUNKS_DIR}"',
        content
    )
    content = re.sub(
        r'CYPHER_DIR = r".*?"',
        f'CYPHER_DIR = r"{CYPHER_DIR}"',
        content
    )
    
    with open(script_path, 'w', encoding='utf-8') as f:
        f.write(content)
    print(f"Updated directories in {script_path}")
else:
    print(f"Could not find {script_path}")

In [ ]:
!python local_extract_cascades.py

## 7. Check Memory Consolidation Stats

In [ ]:
import requests, json
resp = requests.get(f"{CASCADES_URL}/v1/memory/stats")
print(json.dumps(resp.json(), indent=2))

## 8. Save Learned Weights to Drive
After processing, the adapter has absorbed the knowledge into parametric memory.

In [ ]:
import torch, os

SAVE_PATH = "/content/drive/MyDrive/my_digital_twin_weights.pt"

# Extract all trainable adapter weights
adapter_state = {
    name: param.data.cpu().clone()
    for name, param in cascades_model.model.named_parameters()
    if param.requires_grad
}

torch.save(adapter_state, SAVE_PATH)
size_mb = os.path.getsize(SAVE_PATH) / 1024**2
print(f"\n✅ Saved {len(adapter_state)} adapter parameters ({size_mb:.1f} MB) to {SAVE_PATH}")